In [1]:
import json
import random
import numpy as np
import torch

import torch.distributed.fsdp
class FakeFSDPModule:
    pass
torch.distributed.fsdp.FSDPModule = FakeFSDPModule

from datasets import Dataset
from transformers import Trainer, TrainingArguments
from unsloth import FastLanguageModel

/tmp/ipykernel_5424/1910089061.py:13: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


[unsloth_zoo.log|WARNING]Unsloth: Could not patch trl.trainer.grpo_trainer: Direct module loading failed for UnslothGRPOTrainer: Unexpected optimization option triton.enable_persistent_tma_matmul, known options are ['TYPE_CHECKING', 'enable_auto_functionalized_v2', 'debug', 'disable_progress', 'verbose_progress', 'fx_graph_cache', 'fx_graph_remote_cache', 'autotune_local_cache', 'autotune_remote_cache', 'force_disable_caches', 'sleep_sec_TESTING_ONLY', 'custom_op_default_layout_constraint', 'cpp_wrapper', 'abi_compatible', 'c_shim_version', 'dce', 'static_weight_shapes', 'size_asserts', 'nan_asserts', 'pick_loop_orders', 'inplace_buffers', 'allow_buffer_reuse', 'memory_planning', 'memory_pool', 'benchmark_harness', 'epilogue_fusion', 'epilogue_fusion_first', 'pattern_matcher', 'b2b_gemm_pass', 'post_grad_custom_pre_pass', 'post_grad_custom_post_pass', 'joint_custom_pre_pass', 'joint_custom_post_pass', 'pre_grad_custom_pass', '_pre_fusion_custom_pass', 'split_cat_fx_passes', 'efficient_

In [ ]:
MODEL_NAME = "unsloth/llama-3-8b-bnb-4bit"
OUTPUT_DIR = "super_entity_markers_new_hard_test"
SAVE_DIR = "super_entity_markers_new_hard"

MAX_SEQ_LENGTH = 1024
SEED = 3407

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
with open("train_with_hard_dummy_negatives_final_main.jsonl", "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(raw_data)} train examples")

Loaded 8632 train examples


In [ ]:
LABELS = sorted(list(set(x["relation"] for x in raw_data)))
relation_list_str = ", ".join(LABELS)

print("Labels:", LABELS)

Labels: ['ABBREVIATION', 'AFFECTS', 'ALTERNATIVE_NAME', 'APPLIED_TO', 'ASSOCIATED_WITH', 'FINDING_OF', 'HAS_CAUSE', 'ORIGINS_FROM', 'PART_OF', 'PHYSIOLOGY_OF', 'SUBCLASS_OF', 'TO_DETECT_OR_STUDY', 'TREATED_USING', 'USED_IN', 'no_relation']


In [ ]:
MODEL_NAME = "unsloth/llama-3-8b-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.748 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:01<00:00, 263.15it/s]
Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
def insert_typed_entity_markers(text, h_pos, t_pos, h_type, t_type):
    h_start, h_end = h_pos
    t_start, t_end = t_pos

    events = [
        (h_start, 1, (-h_end, 0), f"[HEAD_{h_type}]"),
        (h_end,   0, (-h_start, 1), f"[/HEAD_{h_type}]"),
        (t_start, 1, (-t_end, 1), f"[TAIL_{t_type}]"),
        (t_end,   0, (-t_start, 0), f"[/TAIL_{t_type}]"),
    ]

    events.sort(key=lambda e: (e[0], e[1], e[2]))

    parts = []
    prev = 0
    for pos, _, _, marker in events:
        if pos > prev:
            parts.append(text[prev:pos])
            prev = pos
        parts.append(marker)

    if prev < len(text):
        parts.append(text[prev:])

    return "".join(parts)


def build_prompt(item, relation_list_str):
    """
    Build prompt with:
    - typed entity markers in text
    - explicit head/tail description below text

    Expected item format:
    {
        "text": "...",
        "h": {"name": "...", "pos": [start, end]},
        "t": {"name": "...", "pos": [start, end]},
        "head_type": "...",
        "tail_type": "..."
    }
    """
    text = item["text"]
    h_name = item["h"]["name"]
    t_name = item["t"]["name"]
    h_pos = item["h"]["pos"]
    t_pos = item["t"]["pos"]
    h_type = item.get("head_type", item.get("h", {}).get("type", "UNK"))
    t_type = item.get("tail_type", item.get("t", {}).get("type", "UNK"))

    marked_text = insert_typed_entity_markers(
        text=text,
        h_pos=h_pos,
        t_pos=t_pos,
        h_type=h_type,
        t_type=t_type,
    )

    prompt = f"""You are an expert in biomedical information extraction.
Analyze the text and determine the relation between the two specified entities.
You must choose ONLY ONE relation from the following list:
[{relation_list_str}]

Text:
{marked_text}

Entity 1 (Head): {h_name} (Type: {h_type})
Entity 2 (Tail): {t_name} (Type: {t_type})

Relation:"""

    return prompt


def tokenize_completion_only(example):
    """
    Создаем:
    - input_ids = prompt + " " + label + eos
    - labels = -100 на prompt части, target токены только на answer части
    """

    prompt = build_prompt(example, relation_list_str)
    answer = " " + example["relation"] + tokenizer.eos_token

    # Токенизируем отдельно prompt и answer
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )["input_ids"]

    answer_ids = tokenizer(
        answer,
        add_special_tokens=False,
        truncation=True,
        max_length=64,
    )["input_ids"]

    input_ids = prompt_ids + answer_ids
    attention_mask = [1] * len(input_ids)

    # labels: ignore prompt, predict only answer
    labels = [-100] * len(prompt_ids) + answer_ids

    # Обрезка если вдруг превысили max_seq_length
    if len(input_ids) > MAX_SEQ_LENGTH:
        input_ids = input_ids[:MAX_SEQ_LENGTH]
        attention_mask = attention_mask[:MAX_SEQ_LENGTH]
        labels = labels[:MAX_SEQ_LENGTH]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "prompt_text": prompt,
        "answer_text": answer,
    }

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(tokenize_completion_only)

print("\n--- EXAMPLE PROMPT ---")
print(dataset[0]["prompt_text"])
print("\n--- EXAMPLE ANSWER ---")
print(dataset[0]["answer_text"])
print("----------------------\n")

Map: 100%|██████████| 8632/8632 [00:08<00:00, 1036.22 examples/s]


--- EXAMPLE PROMPT ---
You are an expert in biomedical information extraction.
Analyze the text and determine the relation between the two specified entities.
You must choose ONLY ONE relation from the following list:
[ABBREVIATION, AFFECTS, ALTERNATIVE_NAME, APPLIED_TO, ASSOCIATED_WITH, FINDING_OF, HAS_CAUSE, ORIGINS_FROM, PART_OF, PHYSIOLOGY_OF, SUBCLASS_OF, TO_DETECT_OR_STUDY, TREATED_USING, USED_IN, no_relation]

Text:
Patients with [HEAD_DISO]idiopathic generalized epilepsies[/HEAD_DISO] ([TAIL_DISO]IGE[/TAIL_DISO]) suffered aggravation with CBZ in 17 cases, VPA - in 6, TPM - in 6, LTG - in 1 and LEV - in 1.

Entity 1 (Head): idiopathic generalized epilepsies (Type: DISO)
Entity 2 (Tail): IGE (Type: DISO)

Relation:

--- EXAMPLE ANSWER ---
 ABBREVIATION<|end_of_text|>
----------------------



In [8]:
for i in range(10):
    print("\n--- EXAMPLE PROMPT ---")
    print(dataset[i]["prompt_text"])
    print("\n--- EXAMPLE ANSWER ---")
    print(dataset[i]["answer_text"])
    print("----------------------\n")


--- EXAMPLE PROMPT ---
You are an expert in biomedical information extraction.
Analyze the text and determine the relation between the two specified entities.
You must choose ONLY ONE relation from the following list:
[ABBREVIATION, AFFECTS, ALTERNATIVE_NAME, APPLIED_TO, ASSOCIATED_WITH, FINDING_OF, HAS_CAUSE, ORIGINS_FROM, PART_OF, PHYSIOLOGY_OF, SUBCLASS_OF, TO_DETECT_OR_STUDY, TREATED_USING, USED_IN, no_relation]

Text:
Patients with [HEAD_DISO]idiopathic generalized epilepsies[/HEAD_DISO] ([TAIL_DISO]IGE[/TAIL_DISO]) suffered aggravation with CBZ in 17 cases, VPA - in 6, TPM - in 6, LTG - in 1 and LEV - in 1.

Entity 1 (Head): idiopathic generalized epilepsies (Type: DISO)
Entity 2 (Tail): IGE (Type: DISO)

Relation:

--- EXAMPLE ANSWER ---
 ABBREVIATION<|end_of_text|>
----------------------


--- EXAMPLE PROMPT ---
You are an expert in biomedical information extraction.
Analyze the text and determine the relation between the two specified entities.
You must choose ONLY ONE relati

In [ ]:
class CompletionOnlyCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_token_id = tokenizer.pad_token_id
        if self.pad_token_id is None:
            self.pad_token_id = tokenizer.eos_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)

        input_ids_batch = []
        attention_mask_batch = []
        labels_batch = []

        for f in features:
            input_ids = f["input_ids"]
            attention_mask = f["attention_mask"]
            labels = f["labels"]

            pad_len = max_len - len(input_ids)

            input_ids = input_ids + [self.pad_token_id] * pad_len
            attention_mask = attention_mask + [0] * pad_len
            labels = labels + [-100] * pad_len

            input_ids_batch.append(input_ids)
            attention_mask_batch.append(attention_mask)
            labels_batch.append(labels)

        batch = {
            "input_ids": torch.tensor(input_ids_batch, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask_batch, dtype=torch.long),
            "labels": torch.tensor(labels_batch, dtype=torch.long),
        }
        return batch

data_collator = CompletionOnlyCollator(tokenizer)

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    # warmup_steps=10,
    warmup_ratio=0.1,
    learning_rate=1e-4,
    logging_steps=10,
    save_strategy="epoch",
    # save_steps=50,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,632 | Num Epochs = 3 | Total steps = 405
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 8 x 1) = 64
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)

  2%|▏         | 9/405 [01:11<51:49,  7.85s/it]
                                                 [A
  2%|▏         | 10/405 [01:20<52:43,  8.01s/it]

{'loss': '1.619', 'grad_norm': '4.258', 'learning_rate': '2.195e-05', 'epoch': '0.07414'}



  5%|▍         | 19/405 [02:33<53:03,  8.25s/it]
                                                 [A
  5%|▍         | 20/405 [02:40<50:46,  7.91s/it]

{'loss': '0.6355', 'grad_norm': '3.199', 'learning_rate': '4.634e-05', 'epoch': '0.1483'}



  7%|▋         | 29/405 [03:55<53:40,  8.56s/it]
                                                 [A
  7%|▋         | 30/405 [04:03<52:59,  8.48s/it]

...

{'loss': '0.05163', 'grad_norm': '0.8539', 'learning_rate': '4.396e-06', 'epoch': '2.89'}



 99%|█████████▊| 399/405 [54:07<00:47,  8.00s/it]
                                                 
 99%|█████████▉| 400/405 [54:16<00:40,  8.05s/it]

{'loss': '0.03464', 'grad_norm': '0.8629', 'learning_rate': '1.648e-06', 'epoch': '2.964'}



100%|█████████▉| 404/405 [54:46<00:07,  7.92s/it]
                                                 
100%|██████████| 405/405 [55:07<00:00,  8.17s/it]

{'train_runtime': '3307', 'train_samples_per_second': '7.83', 'train_steps_per_second': '0.122', 'train_loss': '0.1524', 'epoch': '3'}


In [ ]:
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model saved to: {SAVE_DIR}")


Model saved to: super_entity_markers_new_hard


In [ ]:
import json
import torch
import torch.distributed.fsdp
class FakeFSDPModule:
    pass
torch.distributed.fsdp.FSDPModule = FakeFSDPModule
from unsloth import FastLanguageModel
import pandas as pd
from tqdm.auto import tqdm


MODEL_PATH = "super_entity_markers_new_hard"
DEV_PATH = "train_with_hard_dummy_negatives_final_calib.jsonl"

OUTPUT_TSV = "super_entity_markers_new_hard_calib_inf/predictions_rel.tsv"
OUTPUT_CSV = "super_entity_markers_new_hard_calib_inf/predictions_analysis.csv"

DEVICE = "cuda"
MAX_SEQ_LENGTH = 1024
MAX_PROMPT_TOKENS = 900

LABELS = [
    "ABBREVIATION", "ALTERNATIVE_NAME", "SUBCLASS_OF", "PART_OF",
    "TREATED_USING", "ORIGINS_FROM", "TO_DETECT_OR_STUDY", "AFFECTS",
    "HAS_CAUSE", "APPLIED_TO", "USED_IN", "ASSOCIATED_WITH",
    "PHYSIOLOGY_OF", "FINDING_OF",
    "no_relation"
]

relation_list_str = ", ".join(LABELS)


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
model.eval()

dev_data = []
with open(DEV_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            dev_data.append(json.loads(line))

print(f"Loaded {len(dev_data)} examples from {DEV_PATH}")


prompt_template = """You are an expert in biomedical information extraction.
Analyze the text and determine the relation between the two specified entities.
You must choose ONLY ONE relation from the following list:
[{relation_list}]

Text: {text}
Entity 1 (Head): {head_name} (Type: {head_type})
Entity 2 (Tail): {tail_name} (Type: {tail_type})
Relation:"""

def get_doc_id(item):
    return item.get("doc_id", "unknown")

def get_head_name(item):
    return item["h"]["name"]

def get_tail_name(item):
    return item["t"]["name"]

def get_head_type(item):
    return item.get("head_type", item.get("h", {}).get("type", "UNK"))

def get_tail_type(item):
    return item.get("tail_type", item.get("t", {}).get("type", "UNK"))

def get_head_span(item):
    return item.get("head_span", "")

def get_tail_span(item):
    return item.get("tail_span", "")

def get_gold_relation(item):
    return item.get("relation", None)

def insert_typed_entity_markers(text, h_pos, t_pos, h_type, t_type):
    h_start, h_end = h_pos
    t_start, t_end = t_pos

    events = [
        (h_start, 1, (-h_end, 0), f"[HEAD_{h_type}]"),
        (h_end,   0, (-h_start, 1), f"[/HEAD_{h_type}]"),
        (t_start, 1, (-t_end, 1), f"[TAIL_{t_type}]"),
        (t_end,   0, (-t_start, 0), f"[/TAIL_{t_type}]"),
    ]

    events.sort(key=lambda e: (e[0], e[1], e[2]))

    parts = []
    prev = 0
    for pos, _, _, marker in events:
        if pos > prev:
            parts.append(text[prev:pos])
            prev = pos
        parts.append(marker)

    if prev < len(text):
        parts.append(text[prev:])

    return "".join(parts)

def build_prompt(item):
    marked_text = insert_typed_entity_markers(
        text=item["text"],
        h_pos=item["h"]["pos"],
        t_pos=item["t"]["pos"],
        h_type=get_head_type(item),
        t_type=get_tail_type(item),
    )

    return prompt_template.format(
        relation_list=relation_list_str,
        text=marked_text,
        head_name=get_head_name(item),
        head_type=get_head_type(item),
        tail_name=get_tail_name(item),
        tail_type=get_tail_type(item),
    )

label_token_ids = {}
for label in LABELS:
    ids = tokenizer(" " + label, add_special_tokens=False)["input_ids"]
    label_token_ids[label] = ids

@torch.no_grad()
def predict_label_batched(prompt: str, labels):
    """
    Для одного prompt считает все labels одним батчем.
    Без ручного past_key_values.
    """

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_PROMPT_TOKENS,
    )["input_ids"]

    prompt_len = len(prompt_ids)

    batch_input_ids = []
    batch_attention_mask = []
    meta = []

    max_len = 0

    for label in labels:
        suffix_ids = label_token_ids[label]

        full_ids = prompt_ids + suffix_ids

        full_ids = full_ids[:MAX_SEQ_LENGTH]

        actual_suffix_len = max(0, len(full_ids) - prompt_len)

        batch_input_ids.append(full_ids)
        batch_attention_mask.append([1] * len(full_ids))
        meta.append({
            "label": label,
            "prompt_len": prompt_len,
            "suffix_len": actual_suffix_len,
            "suffix_ids": suffix_ids[:actual_suffix_len],
        })

        if len(full_ids) > max_len:
            max_len = len(full_ids)

    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id

    padded_input_ids = []
    padded_attention_mask = []

    for ids, mask in zip(batch_input_ids, batch_attention_mask):
        pad_len = max_len - len(ids)
        padded_input_ids.append(ids + [pad_id] * pad_len)
        padded_attention_mask.append(mask + [0] * pad_len)

    input_ids = torch.tensor(padded_input_ids, dtype=torch.long, device=DEVICE)
    attention_mask = torch.tensor(padded_attention_mask, dtype=torch.long, device=DEVICE)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
        return_dict=True,
    )

    logits = outputs.logits
    log_probs = torch.log_softmax(logits, dim=-1)

    scores = []

    for i, m in enumerate(meta):
        label = m["label"]
        suffix_ids = m["suffix_ids"]
        suffix_len = m["suffix_len"]
        p_len = m["prompt_len"]

        if suffix_len == 0:
            scores.append({
                "label": label,
                "total_logprob": -1e9,
                "avg_logprob": -1e9,
                "num_tokens": 0,
            })
            continue

        total_logprob = 0.0

        for j in range(suffix_len):
            token_pos = p_len + j
            token_id = suffix_ids[j]

            # защита от выхода за границы
            if token_pos - 1 >= logits.shape[1]:
                total_logprob = -1e9
                break

            token_logprob = log_probs[i, token_pos - 1, token_id].item()
            total_logprob += token_logprob

        avg_logprob = total_logprob / max(suffix_len, 1)

        scores.append({
            "label": label,
            "total_logprob": total_logprob,
            "avg_logprob": avg_logprob,
            "num_tokens": suffix_len,
        })

    scores = sorted(scores, key=lambda x: x["avg_logprob"], reverse=True)
    best = scores[0]
    margin = scores[0]["avg_logprob"] - scores[1]["avg_logprob"] if len(scores) > 1 else None

    return best["label"], scores, margin

analysis_rows = []
correct_predictions = 0

with open(OUTPUT_TSV, "w", encoding="utf-8") as tsv_file:
    tsv_file.write("document_id\trelation\thead_text\thead_span\thead_type\ttail_text\ttail_span\ttail_type\n")

    for item in tqdm(dev_data, desc="Batched logprob inference"):
        prompt = build_prompt(item)
        pred_label, scores, margin = predict_label_batched(prompt, LABELS)

        gold_label = get_gold_relation(item)
        is_correct = (pred_label == gold_label)
        if is_correct:
            correct_predictions += 1

        tsv_file.write(
            f"{get_doc_id(item)}\t"
            f"{pred_label}\t"
            f"{get_head_name(item)}\t"
            f"{get_head_span(item)}\t"
            f"{get_head_type(item)}\t"
            f"{get_tail_name(item)}\t"
            f"{get_tail_span(item)}\t"
            f"{get_tail_type(item)}\n"
        )

        row = {
            "document_id": get_doc_id(item),
            "gold_label": gold_label,
            "pred_label": pred_label,
            "is_correct": is_correct,

            "text": item["text"],
            "head_text": get_head_name(item),
            "head_span": get_head_span(item),
            "head_type": get_head_type(item),
            "tail_text": get_tail_name(item),
            "tail_span": get_tail_span(item),
            "tail_type": get_tail_type(item),

            "prompt": prompt,
            "confidence_margin": margin,
            "raw_input_json": json.dumps(item, ensure_ascii=False),
        }

        for s in scores:
            row[f"score_total__{s['label']}"] = s["total_logprob"]
            row[f"score_avg__{s['label']}"] = s["avg_logprob"]
            row[f"num_tokens__{s['label']}"] = s["num_tokens"]

        analysis_rows.append(row)


df = pd.DataFrame(analysis_rows)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")